In [ ]:
!pip install -q langchain langchain-google-genai google-generativeai langgraph langchain-huggingface sentence-transformers accelerate bitsandbytes

In [ ]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
import os

# Suponiendo que tienes una Clave de API almacenada en una variable de entorno
api_key = os.getenv('GEMINI_API_KEY')

# Usa la Clave de API para acceder a un servicio
print(f"Usando la Clave de API: {api_key}")

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGoogleGenerativeAI(
    # model="gemini-2.5-flash",
    # temperature=0,
    # google_api_key=GEMINI_API_KEY
# )


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1
)

hf_llm = HuggingFacePipeline(pipeline=pipe)

# Switch global llm to use local model
llm = hf_llm
print("System switched to local LLM (TinyLlama) to bypass API quota limits.")

In [ ]:
respuesta = llm.invoke("¿Qué es el RAG en Inteligencia Artificial?")
respuesta.content

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Asumiendo que GEMINI_API_KEY ya ha sido definida en una celda anterior.
# Creando una instancia del LLM de Google
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", # Cambiado de "gemini-pro" a "gemini-2.5-flash"
    temperature=0,
    google_api_key=GEMINI_API_KEY
)

# Usando el LLM para generar texto
prompt_text = "Explica qué es inteligencia artificial."
response = llm.invoke([HumanMessage(content=prompt_text)]) # Wrap HumanMessage in a list

# Exibiendo la respuesta generada por el LLM
print(response.content)

In [ ]:
PROMPT_TRIAJE = """
Eres un especialista en triaje del Service Desk para politicas internas.
Dado el mensaje del usuario, devuelve SÓLO un JSON con:\n
{\n
    "decision": "AUTO_RESOLVER" | "PEDIR_INFO" | "ABRIR_TICKET",\n
    "urgency": "BAJA" | "MEDIANA" | "ALTA",\n
    "missing_fields": ["..."]\n
}\n
Reglas:\n
- **AUTO_RESOLVER**: Preguntas claras sobre las reglas o procedimientos descritos en las politicas (Ej.: "¿Puedo reembolsar el internet para mi oficina en casa?").\n
- **PEDIR_INFO**: Mensajes imprecisos o sin información para identificar el tema o el contexto (Ej.: "Necesito ayuda con una politica").\n
- **ABRIR_TICKET**: Solicitudes de excepciones, autorización, aprobación o acceso especial, o cuando el usuario solicita explicitamente abrir un ticket (Ej.: "Quiero una excepción para trabajar remotamente durante 5 dias").\n
Analiza el mensaje y decide la acción más adecuada.
"""

In [ ]:
from typing import Literal, List, Dict

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class TriajeOut(BaseModel):
    decision: Literal["AUTO_RESOLVER", "PEDIR_INFO", "ABRIR_TICKET"]
    urgencia: Literal["BAJA", "MEDIANA", "ALTA"]
    campos_faltantes: List[str] = Field(default_factory=list)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
def triaje(mensaje: str) -> dict:
    prompt = f"{PROMPT_TRIAJE}\n\nMensaje del usuario: {mensaje}\nJSON:"
    try:
        # Invocación al modelo local
        raw_res = hf_llm.invoke(prompt)

        import json
        import re

        # 1. Limpieza: Buscar el primer '{' y el último '}'
        match = re.search(r'\{.*\}', raw_res, re.DOTALL)
        if match:
            json_str = match.group()

            # Normalización para modelos pequeños
            json_str = json_str.replace("'", '"')
            json_str = re.sub(r'(\w+)\s*:', r'"\1":', json_str)
            json_str = re.sub(r',\s*\}', '}', json_str)

            try:
                return json.loads(json_str)
            except json.JSONDecodeError:
                # Fallback: Extraer valores específicos vía Regex
                d_match = re.search(r'"decision"\s*:\s*"(.*?)"', json_str, re.IGNORECASE)
                u_match = re.search(r'"urgenc.*?"\s*:\s*"(.*?)"', json_str, re.IGNORECASE)

                if d_match:
                    return {
                        "decision": d_match.group(1).upper(),
                        "urgencia": u_match.group(1).upper() if u_match else "BAJA",
                        "campos_faltantes": []
                    }

        # Fallback final: Buscar palabras clave en la respuesta cruda
        raw_upper = raw_res.upper()
        if "AUTO_RESOLVER" in raw_upper: return {"decision": "AUTO_RESOLVER", "urgencia": "BAJA", "campos_faltantes": []}
        if "ABRIR_TICKET" in raw_upper: return {"decision": "ABRIR_TICKET", "urgencia": "MEDIANA", "campos_faltantes": []}

        return {"decision": "PEDIR_INFO", "urgencia": "BAJA", "campos_faltantes": []}

    except Exception as e:
        print(f"Error crítico en triaje local: {e}")
        return {"decision": "PEDIR_INFO", "urgencia": "BAJA", "campos_faltantes": []}

In [ ]:
mensajes_de_prueba = [
    "¿Puedo obtener un reembolso por el internet de mi home office?",
    "Quiero una excepción para teletrabajar durante 5 días.",
    "¿Cómo funciona la política de comidas para viajes?",
    "¿Existe una política para anticipos de vacaciones?",
    "¿Quién fue Napoleón Bonaparte?"
]

In [ ]:
for pregunta in mensajes_de_prueba:
    r = triaje(pregunta)
    print(f"{pregunta} -> {r}")

## Describiendo los documentos de recursos humanos

In [ ]:
!pip install -q langchain_community faiss-cpu langchain-text-splitters pymupdf

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

docs = []

for n in Path("/content/").glob("*.pdf"):
    try:
        loader = PyMuPDFLoader(str(n))
        docs.extend(loader.load())
        print(f"Archivo cargado: {n.name}")
    except Exception as e:
        print(f"Error cargando archivo: {n.name}: {e}")

print(f"Total de documentos cargados: {len(docs)}")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

In [ ]:
docs_splits = splitter.split_documents(docs)

In [ ]:
for chunk in docs_splits:
    print(chunk)
    print("-----------------")

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
modelo_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
vectorstore = FAISS.from_documents(docs_splits, modelo_embeddings)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 4}
)

In [ ]:
# Suponemos que tenemos una función para buscar documentos
def buscar_documentos(pregunta):
    # Esta función devuelve documentos relevantes
    documentos = {
        "pandas": "Los pandas se alimentan principalmente de bambú."
    }
    return documentos.get(pregunta.lower(), "")

# Función para generar respuesta usando GPT
def generar_respuesta(pregunta):
    # Primero, recuperamos documentos relevantes
    documento = buscar_documentos(pregunta)

    # Luego, usamos el modelo de IA para generar una respuesta
    if documento:
        respuesta = f"Basado en información interna, sabemos que {documento}"
    else:
        respuesta = "Lo siento, no tengo suficiente información para responder a eso."

    return respuesta

# Ejemplo de uso
pregunta = "¿Qué comen los pandas?"
print(generar_respuesta(pregunta))

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
# from langchain.chains.combine_documents import create_stuff_documents_chain # This import is no longer needed and causes a ModuleNotFoundError

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Simplificamos el prompt para TinyLlama: Menos instrucciones y más directo
prompt_rag = ChatPromptTemplate.from_template("""<|system|>
Usa solo este contexto para responder. Si no está ahí, di 'No lo sé'.
Contexto: {context}</s>
<|user|>
Pregunta: {question}</s>
<|assistant|>
""")

In [ ]:
def busqueda_de_respuestas_RAG(pregunta) -> Dict:
  # The 'retriever' variable in the global scope will now refer to the one
  # associated with the Chroma vector store (defined in cell LsZXp5N89aAB).
  documentos_relacionados = retriever.invoke(pregunta)

  if not documentos_relacionados:
    return {
        "respuesta": "No lo sé",
        "citaciones": [],
        "documentos_encontrados": False
    }

  # Use the 'rag_chain' (defined in cell 3CKgOUx99tVA) which is functional.
  # It handles the context formatting and LLM invocation internally.
  answer = rag_chain.invoke(pregunta)

  if answer.rstrip(".!?") == "No lo sé":
      return {
          "respuesta": "No lo sé",
          "citaciones": [],
          "documentos_encontrados": False
      }

  return {
      "respuesta": answer,
      "citaciones": documentos_relacionados, # Use the separately retrieved documents for citations
      "documentos_encontrados": True
  }

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Definimos la función auxiliar para formatear documentos
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Creamos la cadena RAG necesaria para que la función funcione
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt_rag
    | llm
    | StrOutputParser()
)

# Ejecutamos la búsqueda
resultado = busqueda_de_respuestas_RAG("¿Puedo obtener un reembolso por el internet de mi home office?")
print(resultado)

In [ ]:
mensajes_de_prueba = [
    "¿Puedo obtener un reembolso por el internet de mi home office?",
    "Quiero una excepción para teletrabajar durante 5 días.",
    "¿Cómo funciona la política de comidas para viajes?",
    "¿Existe una politica para anticipos de vacaciones?",
    "¿Quién fue Napoleon Bonaparte?"
]

In [ ]:
# Re-configuramos la cadena con el prompt simplificado y volvemos a probar
if 'hf_llm' in globals():
    llm = hf_llm
    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt_rag
        | llm
        | StrOutputParser()
    )

# Ejecutamos la búsqueda nuevamente
r = busqueda_de_respuestas_RAG("¿Puedo obtener un reembolso por el internet de mi home office?")
print(r['respuesta'])

In [ ]:
print(f"Número de citaciones para la última respuesta: {len(r['citaciones'])}")

In [ ]:
for pregunta in mensajes_de_prueba:
    respuesta_RAG = busqueda_de_respuestas_RAG(pregunta)

    print(f"\nPREGUNTA: {pregunta}")
    print(f"RESPUESTA: {respuesta_RAG['respuesta']}")
    print(f"DOCUMENTOS ENCONTRADOS: {respuesta_RAG['documentos_encontrados']}")

    if respuesta_RAG['documentos_encontrados']:
        for i, citacion in enumerate(respuesta_RAG['citaciones'], start=1):
            print(f"\nCITACION {i}:")
            print(
                f"Camino del documento: "
                f"{citacion.metadata.get('file_path', 'No disponible')}"
            )
            print(
                f"Contenido: "
                f"{citacion.page_content.replace(chr(10), ' ')}"
            )

    print("\n" + "=" * 80)

In [ ]:
import time

# Nos aseguramos de que el rag_chain use el LLM local para evitar errores de cuota
# hf_llm fue definido en la celda fe500b6a
if 'hf_llm' in globals():
    llm = hf_llm

for pregunta in mensajes_de_prueba:
    try:
        print(f"PROCESANDO: {pregunta}")
        respuesta_RAG = busqueda_de_respuestas_RAG(pregunta)

        print(f"PREGUNTA: {pregunta}")
        print(f"RESPUESTA: {respuesta_RAG['respuesta']}")
        print(f"DOCUMENTOS ENCONTRADOS: {respuesta_RAG['documentos_encontrados']}")

        if respuesta_RAG['documentos_encontrados']:
            for i, citacion in enumerate(respuesta_RAG['citaciones']):
                print(f"  CITACION {i + 1}:")
                # Usamos .get() para evitar errores si la clave no existe
                source = citacion.metadata.get('file_path', citacion.metadata.get('source', 'Desconocido'))
                print(f"    Origen: {source}")
                print(f"    Contenido: {citacion.page_content[:200].replace('\n', ' ')}...")

        print("-" * 30)
        # Pequeña pausa para no saturar la salida si el modelo es muy rápido
        time.sleep(0.5)

    except Exception as e:
        print(f"Error al procesar la pregunta '{pregunta}': {e}")

In [ ]:
!apt-get install graphviz -y
!pip install graphviz

In [ ]:
from graphviz import Digraph

g = Digraph("RAG_Flow", format="png")

g.attr(rankdir="TB")

g.node("A", "Usuario")
g.node("B", "Triaje IA")
g.node("C", "Carga PDFs")
g.node("D", "Chunking")
g.node("E", "Embeddings")
g.node("F", "FAISS")
g.node("G", "Retriever")
g.node("H", "Prompt RAG")
g.node("I", "Gemini")
g.node("J", "Respuesta + Citaciones")

g.edge("A", "B")
g.edge("B", "C")
g.edge("C", "D")
g.edge("D", "E")
g.edge("E", "F")
g.edge("F", "G")
g.edge("G", "H")
g.edge("H", "I")
g.edge("I", "J")

g.render("diagrama_rag", view=False)

g

In [ ]:
from graphviz import Digraph

g = Digraph("LangGraph")

g.attr(rankdir="TB")

g.node("START","🚀 START")
g.node("T","🧠 Triaje")

g.node("R","📚 Auto Resolver\n(RAG)")
g.node("I","📋 Pedir Info")
g.node("K","🎫 Abrir Ticket")

g.node("END","✅ END")

g.edge("START","T")

g.edge("T","R","AUTO_RESOLVER")
g.edge("T","I","PEDIR_INFO")
g.edge("T","K","ABRIR_TICKET")

g.edge("R","END","RAG OK")
g.edge("R","I","Sin contexto")
g.edge("R","K","Excepción")

g.edge("I","END")
g.edge("K","END")

g

# OTRO PLANTEAMIENTO

## Instalar dependencias

In [ ]:
# !pip install -q \
# langchain \
# langchain-openai \
# langchain-community \
# langchain-core \
# chromadb \
# pypdf \
# tiktoken \
# langchain-text-splitters # Commented out to prevent dependency conflicts

 ## Configurar API Key

In [ ]:
from google.colab import userdata
import os

os.environ["OPEN_AI_API_KEY"] = userdata.get("OPEN_AI_API_KEY")

## Subir PDFs

In [ ]:
from google.colab import files

uploaded = files.upload()

## Cargar documentos

In [ ]:
# Instalamos la dependencia necesaria para leer PDFs
!pip install -q pypdf

from langchain_community.document_loaders import PyPDFLoader

documents = []

# Cargamos los archivos subidos
for file_name in uploaded.keys():
    try:
        loader = PyPDFLoader(file_name)
        docs = loader.load()
        documents.extend(docs)
    except Exception as e:
        print(f"Error cargando {file_name}: {e}")

print(f"Documentos cargados exitosamente: {len(documents)}")

## Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Chunks creados: {len(chunks)}")

## Crear embeddings

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file_name in uploaded.keys():
    loader = PyPDFLoader(file_name)
    docs = loader.load()

    documents.extend(docs)

print(f"Documentos cargados: {len(documents)}")

In [ ]:
!pip install -q langchain-openai

from langchain_openai import OpenAIEmbeddings
from google.colab import userdata
from google.colab.userdata import SecretNotFoundError
import os

try:
    openai_api_key = userdata.get("OPEN_AI_API_KEY")
    # Establecemos la variable de entorno estándar para LangChain
    os.environ["OPENAI_API_KEY"] = openai_api_key
except SecretNotFoundError:
    raise ValueError(
        "OpenAI API key not found. Please ensure you have set a Colab secret named 'OPEN_AI_API_KEY'."
    )

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

### Usar Embeddings de HuggingFace (gratuitos y locales)

In [ ]:
# Instalar las librerías necesarias para HuggingFace Embeddings y la versión recomendada por LangChain
# !pip install -q --force-reinstall langchain-huggingface sentence-transformers transformers # Commented out to prevent dependency conflicts

### Instalar librerías adicionales para ejecutar modelos locales (HuggingFace LLM)

In [ ]:
# `accelerate` y `bitsandbytes` son útiles para ejecutar modelos más grandes de forma eficiente.
# `torch` debería estar instalado por otras dependencias, pero asegúrate de que sea una versión compatible.
# !pip install -q accelerate bitsandbytes # Commented out to prevent dependency conflicts

In [ ]:
# Importar HuggingFaceEmbeddings de la nueva ubicación recomendada
from langchain_huggingface import HuggingFaceEmbeddings

# Inicializar HuggingFaceEmbeddings con un modelo pre-entrenado. Puedes elegir otros modelos si lo deseas.
# 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' es un buen modelo multilingüe.
# Para modelos en español, considera 'hiiamsid/sentence_similarity_spanish_es'
# O 'distilbert-base-nli-stsb-mean-tokens' para inglés (requiere descargar el modelo)

hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Asignar la instancia de HuggingFaceEmbeddings a la variable 'embeddings'
# para que el siguiente paso de Chroma.from_documents la use.
embeddings = hf_embeddings

print("Embeddings de HuggingFace inicializados correctamente.")

In [ ]:
# Solución al conflicto de dependencias y error de atributo en Colab
!pip install -q chromadb opentelemetry-api==1.25.0 opentelemetry-sdk==1.25.0

import chromadb
from chromadb.config import Settings
from langchain_community.vectorstores import Chroma

# Crear la base vectorial usando Chroma con la configuración corregida
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",
    client_settings=Settings(is_persistent=True, persist_directory="./chroma_db")
)

print("Base vectorial creada exitosamente con Chroma")

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [ ]:
query = "¿Cuál es el objetivo principal del documento?"

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---\n")
    print(doc.page_content[:500])

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Eres un asistente especializado en responder preguntas
usando exclusivamente el contexto proporcionado.

Reglas:

1. Responde únicamente con información presente en el contexto.
2. No inventes datos.
3. Si la respuesta no está en el contexto, responde:

"No encontré información suficiente en los documentos para responder esa pregunta."

Contexto:

{context}

Pregunta:

{question}

Respuesta:
""")

In [ ]:
# Actualizamos a versiones recientes y estables para corregir el error de compatibilidad _add_version
!pip install -q -U langchain-openai langchain-core

import os
from langchain_openai import ChatOpenAI

# Aseguramos que la API Key esté disponible
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    from google.colab import userdata
    openai_api_key = userdata.get("OPEN_AI_API_KEY")
    os.environ["OPENAI_API_KEY"] = openai_api_key

try:
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        api_key=openai_api_key
    )
    print("LLM de OpenAI (GPT-4o-mini) inicializado correctamente.")
except Exception as e:
    print(f"Error al inicializar el LLM: {e}")
    print("Si el error persiste, intenta reiniciar el entorno (Runtime -> Restart session).")

### Configurar un LLM de Hugging Face

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
import torch

# Define el nombre del modelo de Hugging Face que quieres usar
# TinyLlama/TinyLlama-1.1B-Chat-v1.0 es un modelo pequeño adecuado para pruebas en Colab
# Si tienes GPU y más RAM, puedes probar modelos más grandes como 'google/gemma-2b-it'
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Cargar el tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Cargar el modelo. Usa torch_dtype=torch.bfloat16 para mayor eficiencia si tu GPU lo soporta.
# Si tienes problemas de memoria, puedes intentar cargar con quantization_config (e.g., bitsandbytes)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

# Crear un pipeline de texto con el modelo y el tokenizer
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.1,
    do_sample=False
    # max_new_tokens=512, # Limita la longitud de la respuesta generada
    # temperature=0.1,    # Ajusta para controlar la creatividad de la respuesta
)

# Inicializar el LLM de HuggingFace para LangChain
hf_llm = HuggingFacePipeline(pipeline=pipeline)

# Sobrescribir la variable `llm` para que el `rag_chain` use el nuevo LLM de Hugging Face
llm = hf_llm

print(f"LLM de Hugging Face '{model_name}' inicializado correctamente y configurado para la cadena RAG.")


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
# The format_docs function definition was moved to cell 3CKgOUx99tVA to ensure it's defined before rag_chain.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt_rag # Changed prompt to prompt_rag
    | llm  # This will now correctly reference the HuggingFace LLM
    | StrOutputParser()
)

In [ ]:
docs = retriever.invoke(
    "¿Cada cuánto deben cambiarse las contraseñas?"
)

print(format_docs(docs))

In [ ]:
rag_chain.invoke(
    "¿Cuál es la política de licencia por maternidad?"
)

```mermaid
flowchart TD

A[Usuario] --> B[Agente de Triaje]

B --> C[Carga de PDFs]

C --> D[Chunking (RecursiveCharacter)]

D --> E[Embeddings (HuggingFace/OpenAI)]

E --> F[Vector Store (Chroma/FAISS)]

F --> G[Retriever]

G --> H[Prompt RAG]

H --> I[LLM (OpenAI GPT-4o-mini / TinyLlama)]

I --> J[Respuesta Final]

J --> K[Citaciones de Documentos]
```

In [ ]:
!pip install -q langgraph

In [ ]:
from typing import TypedDict, Optional
class AgentState(TypedDict, total=False):
      pregunta: str
      triaje: dict
      respuesta: Optional[str]
      citaciones: Optional[list]
      documentos_encontrados: Optional[bool]
      rag_exito: bool
      accion_final: str

In [ ]:
from langgraph.graph import START, END, StateGraph
from typing import TypedDict

# --- EXPLICACIÓN DEL ERROR ---
# El error 'NameError' ocurría porque 'nodo_triaje' y otras funciones
# se estaban llamando antes de ser definidas.
# Para usar LangGraph, primero define tus funciones de nodo.

# workflow = StateGraph(AgentState)
# workflow.add_node("triaje", nodo_triaje)
# ... etc

# --- EJEMPLO DE TYPEDDICT (ESTO SÍ FUNCIONA) ---

# Definición del diccionario tipado 'Alumno'
class Aluno(TypedDict):
    nombre: str  # El nombre debe ser un string
    edad: int # La edad debe ser un entero (int)
    notas: list # Las notas deben ser una lista

# Creando un diccionario del tipo Alumno
alumno1: Aluno = {
    "nombre": "Juan",
    "edad": 20,
    "notas": [8.5, 9.0, 7.5]
}

print(f"Alumno cargado: {alumno1['nombre']}")

# Esto causaría un error de tipo (comprobado por herramientas externas como MyPy)
# alumno_errado: Aluno = {
#     "nome": "Maria",
#     "idade": "vinte",  # Error: idade (edad) debe ser un int
#     "notas": [9.0, 8.0, 7.0]
# }

In [ ]:
from typing import Optional

# La función acepta 'telefono'  como string o None
def saludo(nombre: str, telefono: Optional[str] = None) -> str:
    if telefono:
        return f"Hola, {nombre}! Tu teléfono es {telefono}."
    else:
        return f"Hola, {nombre}!"

# Usando la función con y sin teléfono
print(saludo("Ana"))  # Salida: Hola, Ana!
print(saludo("Carlos", "1234-5678"))  # Salida: Hola, Carlos! Tu teléfono es 1234-5678.

In [ ]:
from langgraph.graph import START, END, StateGraph

# 1. Definición de los Nodos (Funciones)
def nodo_triage(state: AgentState) -> AgentState:
    print("Ejecutando nodo triaje...")
    return {"triaje": triaje(state["pregunta"])}

def nodo_auto_resolver(state: AgentState) -> AgentState:
    print("Ejecutando nodo 'auto_resolver'...")
    respuesta_RAG = busqueda_de_respuestas_RAG(state["pregunta"])
    update = {
        "respuesta": respuesta_RAG["respuesta"],
        "citaciones": respuesta_RAG["citaciones"],
        "rag_exito": respuesta_RAG["documentos_encontrados"]
    }
    update["accion_final"] = "AUTO_RESOLVER" if respuesta_RAG["documentos_encontrados"] else "pedir_info"
    return update

def nodo_pedir_info(state: AgentState) -> AgentState:
    print("Ejecutando nodo 'pedir_info'...")
    return {
        "respuesta": "Necesito más información sobre tu pedido.",
        "citaciones": [],
        "accion_final": "PEDIR_INFO"
    }

# 2. Definición de las Aristas (Lógica de decisión)
def arista_decision_triaje(state: AgentState):
    decision = state.get("triaje", {}).get("decision")
    if decision == "AUTO_RESOLVER":
        return "rag"
    elif decision == "PEDIR_INFO":
        return "info"
    else:
        return "ticket"

def arista_decision_rag(state: AgentState):
    if state.get("rag_exito"):
        return "ok"
    return "info" if state.get("accion_final") == "pedir_info" else "ticket"

# 3. Construcción del Grafo
workflow = StateGraph(AgentState)

workflow.add_node("triaje", nodo_triage)
workflow.add_node("auto_resolver", nodo_auto_resolver)
workflow.add_node("pedir_info", nodo_pedir_info)
workflow.add_node("abrir_ticket", lambda state: {"respuesta": "He abierto un ticket con RR.HH.", "accion_final": "TICKET"})

workflow.add_edge(START, "triaje")

workflow.add_conditional_edges("triaje", arista_decision_triaje, {
    "rag": "auto_resolver",
    "info": "pedir_info",
    "ticket": "abrir_ticket"
})

workflow.add_conditional_edges("auto_resolver", arista_decision_rag, {
    "info": "pedir_info",
    "ticket": "abrir_ticket",
    "ok": END
})

workflow.add_edge("pedir_info", END)
workflow.add_edge("abrir_ticket", END)

# 4. Compilación
grafo = workflow.compile()
print("Grafo reiniciado, nodos definidos y compilado exitosamente.")

In [ ]:
def nodo_triage(state: AgentState) -> AgentState:
    print("Ejecutando nodo triaje...")
    return {"triaje": triaje(state["pregunta"])}

def nodo_auto_resolver(state: AgentState) -> AgentState:
    print("Ejecutando nodo 'auto_resolver'...")
    respuesta_RAG = busqueda_de_respuestas_RAG(state["pregunta"])

    update: AgentState = {
        "respuesta": respuesta_RAG["respuesta"],
        "citaciones": respuesta_RAG["citaciones"],
        "rag_exito": respuesta_RAG["documentos_encontrados"]
    }

    if respuesta_RAG["documentos_encontrados"]:
        update["accion_final"] = "AUTO_RESOLVER"
    else:
        update["accion_final"] = "pedir_info"

    return update

In [ ]:
def nodo_pedir_info(state: AgentState) -> AgentState:
    print("Ejecutando nodo 'pedir_info'...");
    return {
        "respuesta": "Necesito más informaciones sobre tu pedido.",
        "citaciones": [],
        "accion_final": "PEDIR_INFO"
    }

In [ ]:
def nodo_abrir_ticket(state: AgentState) -> AgentState:
    print("Ejecutando nodo 'abrir_ticket'...");
    tri = state["triage"];
    return {
        "respuesta": f"Abrir ticket con urgencia {tri['urgencia']}. Pedido: {state['pregunta']}.",
        "citaciones": [],
        "accion_final": "ABRIR_TICKET"
    }

In [ ]:
def arista_decision_triaje(state: AgentState) -> str:
    print("Decidiendo el flujo después del nodo 'triaje'...")
    tri = state["triaje"]

    if tri["decision"] == "AUTO_RESOLVER":
        return "rag"
    elif tri["decision"] == "PEDIR_INFO":
        return "info"
    else:
        return "ticket"

In [ ]:
def arista_decision_rag(state: AgentState) -> str:
    print("Decidiendo el flujo después del nodo 'auto_resolver'...")
    if state["rag_exito"]:
        print("RAG con exito, finalizando el flujo.")
        return "ok"

In [ ]:
def arista_decision_rag(state: AgentState) -> str:
    print("Decidiendo el flujo después del nodo 'auto_resolver'...")

    # 1. Si el RAG tuvo éxito, terminamos
    if state.get("rag_exito"):
        print("RAG con éxito, finalizando el flujo.")
        return "ok"

    # 2. Si falló el RAG, revisamos palabras clave para decidir si abrir ticket o pedir info
    KEYWORDS_ABRIR_TICKET = ["aprobación", "aprobar", "excepción", "liberación", "autorización",
                             "autorizar", "abrir ticket", "acceso especial"]

    pregunta_usuario = state.get("pregunta", "").lower()

    if any(keyword in pregunta_usuario for keyword in KEYWORDS_ABRIR_TICKET):
        print("Pregunta relacionada con excepciones/permisos, redirigiendo a 'ticket'.")
        return "ticket"

    print("RAG ha fallado y no es una solicitud de excepción, pediré más informaciones al usuario.")
    return "info"

In [ ]:
import time

for pregunta in mensajes_de_prueba:
    print(f"\n--- PROCESANDO: {pregunta} ---")
    estado_inicial = {"pregunta": pregunta}

    try:
        # Ejecutamos el grafo en modo stream para ver el paso a paso
        for event in grafo.stream(estado_inicial):
            for node_name, state_update in event.items():
                print(f"\n[Nodo: {node_name}]")

                # Mostramos la decisión si el nodo es triaje
                if "triaje" in state_update:
                    tri = state_update["triaje"]
                    # Manejamos ambos posibles formatos de la respuesta del modelo
                    decision = tri.get("decision") or tri.get("triaje", {}).get("decision")
                    print(f"Decisión: {decision}")

                # Mostramos la respuesta final si existe
                if "respuesta" in state_update:
                    print(f"Respuesta: {state_update['respuesta']}")

        # Pausa de seguridad para evitar bloqueos por cuota de API
        time.sleep(2)

    except Exception as e:
        print(f"\n[ERROR] Falló el procesamiento de la pregunta: {e}")
        print("Reintentando en el siguiente ciclo...")
        time.sleep(5)

    print("\n" + "="*50)

In [ ]:
from IPython.display import display, Image

graph_bytes = grafo.get_graph().draw_mermaid_png()
display(Image(graph_bytes))

In [ ]:
PREGUNTA = "¿Puedo reembolsar mi internet?"

try:
    # Now using the local llm configured in the previous step
    temp = grafo.invoke({"pregunta": PREGUNTA})

    print(f"PREGUNTA: {PREGUNTA}")
    triaje_data = temp.get('triaje', {})
    decision = triaje_data.get('decision', 'N/A')
    urgencia = triaje_data.get('urgencia', 'N/A')

    print(f"DECISION: {decision} | URGENCIA: {urgencia}")
    print(f"RESPUESTA: {temp.get('respuesta', 'Sin respuesta')}")

except Exception as e:
    print(f"Error durante la ejecución: {e}")

In [ ]:
mensajes_de_prueba = [
	"¿Puedo obtener un reembolso por el internet de mi home office?",
	"Quiero una excepción para teletrabajar durante 5 días.",
	"¿Cómo funciona la política de comidas para viajes?",
	"¿Existe una política para anticipos de vacaciones?",
	"¿Quién fue Napoleón Bonaparte?"
]

In [ ]:
import time

for prueba in mensajes_de_prueba:
  try:
    print(f"\n--- Procesando: {prueba} ---")
    # El grafo ahora usará las funciones de nodo actualizadas con hf_llm
    respuesta = grafo.invoke({"pregunta": prueba})

    triaje_data = respuesta.get('triaje', {})
    decision = triaje_data.get('decision', 'N/A')
    urgencia = triaje_data.get('urgencia', 'N/A')

    print(f"DECISIÓN: {decision} | URGENCIA: {urgencia} | ACCIÓN FINAL: {respuesta.get('accion_final', 'N/A')}")
    print(f"RESPUESTA: {respuesta.get('respuesta', 'Sin respuesta')}")

    if respuesta.get('citaciones'):
        print(f"[Citaciones encontradas: {len(respuesta['citaciones'])}]")

    print("-----------------------------------------------")

    # Tiempo de espera más largo para seguridad del entorno
    print("Esperando 20 segundos para liberar recursos...")
    time.sleep(20)

  except Exception as e:
    print(f"Error procesando '{prueba}': {e}")
    time.sleep(10)

In [ ]:
from graphviz import Digraph

g = Digraph("HR_AI_System", format="png")

g.attr(
    rankdir="TB",
    bgcolor="white",
    fontname="Helvetica",
    splines="ortho"
)

# Estilos
g.attr("node",
       shape="box",
       style="rounded,filled",
       color="#1F4E79",
       fillcolor="#DCE6F1",
       fontname="Helvetica")

# Flujo principal

g.node("USER", "👤 Usuario")

g.node("TRIAGE",
       "🧠 Agente de Triaje\nTinyLlama / Gemini")

g.node("INFO",
       "📋 Pedir Información")

g.node("TICKET",
       "🎫 Abrir Ticket")

g.node("RAG",
       "📚 Motor RAG")

g.node("RETR",
       "🔍 Retriever\nFAISS / Chroma")

g.node("EMB",
       "🧬 Embeddings")

g.node("PDF",
       "📄 PDFs")

g.node("LLM",
       "🤖 Gemini / TinyLlama")

g.node("RESP",
       "✅ Respuesta\n+ Citaciones")

# Conexiones

g.edge("USER","TRIAGE")

g.edge("TRIAGE","INFO",
       label="PEDIR_INFO")

g.edge("TRIAGE","TICKET",
       label="ABRIR_TICKET")

g.edge("TRIAGE","RAG",
       label="AUTO_RESOLVER")

g.edge("RAG","RETR")

g.edge("PDF","EMB")
g.edge("EMB","RETR")

g.edge("RETR","LLM")
g.edge("LLM","RESP")

g